In [1]:
# --- 1. Imports & NLTK setup ---
import pandas as pd
import numpy as np
import re

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

pd.set_option('display.max_colwidth', 150)

# --- 2. Load the dataset ---
df = pd.read_csv('IMDB_Dataset_CLEANED.csv')
print("Shape:", df.shape)
print(df.head())

# --- 3. Inspect missing values and duplicates ---
print("\nMissing values per column:")
print(df.isnull().sum())

num_duplicates = df.duplicated(subset='review').sum()
print(f"\nNumber of duplicate reviews: {num_duplicates}")

print("\nSentiment value counts:")
print(df['sentiment'].value_counts())

# Drop missing values and duplicate reviews
df = df.dropna(subset=['review'])
df = df.drop_duplicates(subset='review').reset_index(drop=True)
print("\nShape after cleaning missing/duplicates:", df.shape)

# --- 4. Text cleaning function ---
# (lowercase -> remove HTML/URLs/numbers/punctuation -> tokenize -> remove stopwords -> stem/lemmatize)

USE_LEMMATIZATION = True   # set to False if you want stemming instead

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()                                    # Step: lowercase
    text = re.sub(r'<.*?>', ' ', text)                      # remove HTML tags
    text = re.sub(r'http\S+|www\.\S+', ' ', text)           # remove URLs
    text = re.sub(r'\d+', ' ', text)                        # remove numbers
    text = re.sub(r'[^a-z\s]', ' ', text)                   # remove punctuation/special chars
    text = re.sub(r'\s+', ' ', text).strip()                 # collapse extra whitespace

    tokens = word_tokenize(text)                             # tokenize
    tokens = [tok for tok in tokens if tok not in stop_words and len(tok) > 1]  # remove stopwords

    if USE_LEMMATIZATION:
        tokens = [lemmatizer.lemmatize(tok) for tok in tokens]  # lemmatize
    else:
        tokens = [stemmer.stem(tok) for tok in tokens]          # stem

    return ' '.join(tokens)

# quick sanity test
sample = "This movie was AMAZING!!! I'd watch it 10 times. Visit http://example.com <br /><br /> 5/10 overall."
print("\nOriginal:", sample)
print("Cleaned :", clean_text(sample))

# --- 5. Apply cleaning to the full dataset ---
from tqdm import tqdm
tqdm.pandas()

df['cleaned_review'] = df['review'].progress_apply(clean_text)
print(df[['review', 'cleaned_review', 'sentiment']].head())

# --- 6. Compare original vs cleaned reviews ---
pd.set_option('display.max_colwidth', 300)

for i in range(3):
    print(f"\n--- Example {i+1} ---")
    print("Original :", df['review'].iloc[i][:300])
    print("Cleaned  :", df['cleaned_review'].iloc[i][:300])
    print("Sentiment:", df['sentiment'].iloc[i])

# optional word count sanity check
df['original_word_count'] = df['review'].apply(lambda x: len(x.split()))
df['cleaned_word_count'] = df['cleaned_review'].apply(lambda x: len(x.split()))
print("\n", df[['original_word_count', 'cleaned_word_count']].describe())

# --- 7. Export the preprocessed dataset ---
output_columns = ['review', 'sentiment', 'cleaned_review']
df[output_columns].to_csv('IMDB_Dataset_PREPROCESSED.csv', index=False)

print("\nSaved: IMDB_Dataset_PREPROCESSED.csv")
print("Final shape:", df[output_columns].shape)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kirut\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kirut\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kirut\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\kirut\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\kirut\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Shape: (49396, 2)
                                                                                                                                                  review  \
0  $25,000 Pyramid Clues: Deep Blue Sea. Tremors. Slither. Eight Legged Freaks.<br /><br />Pyramid Category: Movies that were funnier and more thrill...   
1  0.5/10. This movie has absolutely nothing good about it. The acting is among the worst I have ever seen, what is really amazing is that EVERYONE i...   
2  0*'s Christian Slater, Tara Reid, Stephen Dorff, Frank C. Turner, Mathew Walker, Will Sanderson. Directed by Uwe Boll.<br /><br />Based on the vid...   
3  102 Dalmatians (2000, Dir. Kevin Lima) <br /><br />Believed to be cured, Cruella de Vil (Close) is released from prison and sets out to make a new...   
4  102 DALMATIANS [Walt Disney]: I wasn't a fan of the previous installment and this effort has all the weaknesses of the first, a silly padded story...   

  sentiment  
0  negative  
1  negative  
2  

100%|██████████| 49396/49396 [01:07<00:00, 736.81it/s]


                                                                                                                                                  review  \
0  $25,000 Pyramid Clues: Deep Blue Sea. Tremors. Slither. Eight Legged Freaks.<br /><br />Pyramid Category: Movies that were funnier and more thrill...   
1  0.5/10. This movie has absolutely nothing good about it. The acting is among the worst I have ever seen, what is really amazing is that EVERYONE i...   
2  0*'s Christian Slater, Tara Reid, Stephen Dorff, Frank C. Turner, Mathew Walker, Will Sanderson. Directed by Uwe Boll.<br /><br />Based on the vid...   
3  102 Dalmatians (2000, Dir. Kevin Lima) <br /><br />Believed to be cured, Cruella de Vil (Close) is released from prison and sets out to make a new...   
4  102 DALMATIANS [Walt Disney]: I wasn't a fan of the previous installment and this effort has all the weaknesses of the first, a silly padded story...   

                                                               